# nano-llm Kaggle runner

Thin launcher: clone the repo, install pinned deps, run `orchestrate.py`.

**Before pressing Run All:**
- Settings → Accelerator: **GPU T4 x2**
- Settings → Internet: **On** (needed for git clone + HF datasets)
- (Optional, for resume) Attach a Kaggle Dataset that holds your previous `checkpoints/` folder under Add Input → Datasets.

**Always start with the smoke config** (cell 3a) to confirm the pipeline works end-to-end in ~2–3 min before kicking off the real run.

In [1]:
# 1. Clone + install
!rm -rf nano-llm && git clone https://github.com/ferhat00/nano-llm.git
%cd nano-llm
!pip install -q -r requirements.txt

Cloning into 'nano-llm'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 56 (delta 19), reused 46 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 38.06 KiB | 3.17 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/kaggle/working/nano-llm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 47.5 MB/s eta 0:00:00
  

In [2]:
# 2. (OPTIONAL) Resume from a previous session's checkpoints.
# After a session ends, save the checkpoints folder as a Kaggle Dataset, then
# attach it here under Add Input and uncomment the lines below.
#
# !mkdir -p /kaggle/working/checkpoints
# !cp -a /kaggle/input/<your-dataset-slug>/checkpoints/. /kaggle/working/checkpoints/
# !ls /kaggle/working/checkpoints

## 3a. Smoke test (always run this first)

In [3]:
# Smoke runs CPU by default. Override device=cuda to exercise the GPU/AMP code path quickly.
!python orchestrate.py --config configs/smoke.yaml --train.device=cuda --train.precision=fp16

  nano-llm
torch=2.4.1+cu121  cuda_available=True
cuda devices (2):
  [0] Tesla T4  vram=14.6 GB  cc=7.5
  [1] Tesla T4  vram=14.6 GB  cc=7.5

--- resolved config ---
seed: 1337
out_dir: ./_smoke_run
tokenizer:
  type: bpe
  vocab_size: 2048
  train_subset_docs: 200
data:
  dataset: roneneldan/TinyStories
  text_field: text
  subset_train: 1000
  subset_val: 200
model:
  n_layer: 2
  n_head: 4
  n_embd: 64
  block_size: 128
  dropout: 0.0
train:
  device: cuda
  precision: fp16
  distributed: false
  batch_size: 8
  grad_accum: 1
  max_steps: 30
  learning_rate: 0.0003
  min_lr: 3.0e-05
  beta1: 0.9
  beta2: 0.95
  weight_decay: 0.1
  grad_clip: 1.0
  warmup_steps: 5
  eval_interval: 10
  eval_iters: 5
  sample_interval: 30
  checkpoint_interval: 10
  keep_last_k: 3
sample:
  prompt: Once upon a time
  max_new_tokens: 40
  temperature: 0.9
  top_k: 40
  top_p: 0.9

------------------------------------------------------------------------
  STAGE 1 / tokenizer
---------------------------

## 3b. Real run (~30M params on TinyStories)

In [4]:
!python orchestrate.py --config configs/small.yaml

  nano-llm
torch=2.4.1+cu121  cuda_available=True
cuda devices (2):
  [0] Tesla T4  vram=14.6 GB  cc=7.5
  [1] Tesla T4  vram=14.6 GB  cc=7.5

--- resolved config ---
seed: 1337
out_dir: /kaggle/working/checkpoints
tokenizer:
  type: bpe
  vocab_size: 8192
  train_subset_docs: 50000
data:
  dataset: roneneldan/TinyStories
  text_field: text
model:
  n_layer: 8
  n_head: 8
  n_embd: 512
  block_size: 512
  dropout: 0.1
  rope_base: 10000.0
train:
  device: cuda
  precision: fp16
  distributed: true
  batch_size: 32
  grad_accum: 8
  max_steps: 2000
  learning_rate: 0.0006
  min_lr: 6.0e-05
  beta1: 0.9
  beta2: 0.95
  weight_decay: 0.1
  grad_clip: 1.0
  warmup_steps: 200
  eval_interval: 200
  eval_iters: 50
  sample_interval: 500
  checkpoint_interval: 500
  keep_last_k: 3
sample:
  prompt: Once upon a time, there was a little
  max_new_tokens: 200
  temperature: 0.8
  top_k: 40
  top_p: 0.9
huggingface:
  repo_id: ferhat00/nano-llm-tinystories
  private: false

----------------------

## 4. Push the trained model to Hugging Face

Uploads the latest checkpoint (`pytorch_model.pt`), the tokenizer, and an
auto-generated model card to your HF account. The repo is created if it doesn't
exist.

**Before running:**
- Create a Hugging Face **write** token: huggingface.co → Settings → Access Tokens.
- Add it as a Kaggle Secret named **`HF_TOKEN`**: Add-ons → Secrets → Add a new secret.
- In cell 4b below, set `repo_id` to `your-username/nano-llm-tinystories`.

In [5]:
# 4a. Load the HF write token from Kaggle Secrets into the environment.
# IPython propagates os.environ to the `!python ...` subprocess in the next cell.
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))

HF_TOKEN loaded: True


In [6]:
# 4b. Push. Set repo_id to your HF username/repo.
!python orchestrate.py --stage push --config configs/small.yaml \
    --huggingface.repo_id=ferhat00/nano-llm-tinystories

  nano-llm
torch=2.4.1+cu121  cuda_available=True
cuda devices (2):
  [0] Tesla T4  vram=14.6 GB  cc=7.5
  [1] Tesla T4  vram=14.6 GB  cc=7.5

--- resolved config ---
seed: 1337
out_dir: /kaggle/working/checkpoints
tokenizer:
  type: bpe
  vocab_size: 8192
  train_subset_docs: 50000
data:
  dataset: roneneldan/TinyStories
  text_field: text
model:
  n_layer: 8
  n_head: 8
  n_embd: 512
  block_size: 512
  dropout: 0.1
  rope_base: 10000.0
train:
  device: cuda
  precision: fp16
  distributed: true
  batch_size: 32
  grad_accum: 8
  max_steps: 2000
  learning_rate: 0.0006
  min_lr: 6.0e-05
  beta1: 0.9
  beta2: 0.95
  weight_decay: 0.1
  grad_clip: 1.0
  warmup_steps: 200
  eval_interval: 200
  eval_iters: 50
  sample_interval: 500
  checkpoint_interval: 500
  keep_last_k: 3
sample:
  prompt: Once upon a time, there was a little
  max_new_tokens: 200
  temperature: 0.8
  top_k: 40
  top_p: 0.9
huggingface:
  repo_id: ferhat00/nano-llm-tinystories
  private: false

----------------------